# Lesson 4A: Support Vector Machines Theory

<a name="introduction"></a>
## Introduction

Imagine you're designing a security system to separate authorized from unauthorized access. You want the decision boundary to be as far as possible from both classes — the farther away, the more robust your system is to new visitors.

Support Vector Machines (SVMs) are built on this principle: find the boundary that maximizes the margin (distance) between classes. This elegant idea leads to a powerful classifier that works remarkably well in high dimensions and for non-linear problems.

What makes SVMs special:
1. **Maximum margin principle**: We don't just separate classes; we find the widest separation possible
2. **Kernel trick**: We can handle non-linear patterns by computing dot products in high-dimensional spaces without ever explicitly computing those high-dimensional transformations
3. **Sparsity**: Only a few data points (support vectors) determine the decision boundary
4. **Theoretical foundation**: SVMs have strong generalization guarantees grounded in statistical learning theory

In this lesson, we'll:
1. Understand the geometric intuition behind maximum margin classification
2. Derive the optimization problem mathematically using Lagrangian duality
3. Introduce the kernel trick to handle non-linear separation
4. Implement SVM from scratch using gradient descent on the hinge loss
5. Visualize decision boundaries and support vectors
6. Apply SVMs to real-world classification problems

Then in Lesson 4b, we'll use Scikit-learn's highly optimized SVM implementation and explore advanced kernels and techniques.

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [The Maximum Margin Principle](#the-maximum-margin-principle)
   - [Geometric Intuition](#geometric-intuition)
   - [Mathematical Formulation](#mathematical-formulation)
   - [The Margin and Support Vectors](#the-margin-and-support-vectors)
4. [Lagrangian Duality](#lagrangian-duality)
   - [Primal Problem](#primal-problem)
   - [Lagrangian and KKT Conditions](#lagrangian-and-kkt-conditions)
   - [Dual Problem](#dual-problem)
5. [The Kernel Trick](#the-kernel-trick)
   - [Non-Linear Separation](#non-linear-separation)
   - [Kernel Functions](#kernel-functions)
   - [Mercer's Theorem](#mercers-theorem)
6. [Soft-Margin SVM](#soft-margin-svm)
   - [Allowing Misclassification](#allowing-misclassification)
   - [Hinge Loss](#hinge-loss)
7. [From-Scratch Implementation](#from-scratch-implementation)
   - [Linear SVM with Gradient Descent](#linear-svm-with-gradient-descent)
   - [Non-Linear SVM with Kernels](#non-linear-svm-with-kernels)
8. [Visualization and Interpretation](#visualization-and-interpretation)
   - [Decision Boundaries](#decision-boundaries)
   - [Support Vectors](#support-vectors)
   - [Margin Visualization](#margin-visualization)
9. [Practical Application](#practical-application)
   - [Binary Classification on Breast Cancer Data](#binary-classification-on-breast-cancer-data)
   - [Performance Analysis](#performance-analysis)
10. [Conclusion](#conclusion)
    - [Key Insights](#key-insights)
    - [When to Use SVMs](#when-to-use-svms)
    - [Further Reading](#further-reading)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

<a name="the-maximum-margin-principle"></a>
## The Maximum Margin Principle

<a name="geometric-intuition"></a>
### Geometric Intuition

Consider a simple binary classification problem: we have two groups of points and want to draw a line between them. Many lines could separate them, but which is best?

**Maximum margin principle**: Choose the line that has the maximum distance to the nearest points on either side. These nearest points are called **support vectors**.

Why does this work?
- A boundary far from all points is more robust to new data
- It generalizes better because it's not overfitting to individual data points
- It has strong theoretical guarantees on generalization

In [ ]:
# Visualize the maximum margin principle
np.random.seed(42)

# Generate separable data
X_class0 = np.random.randn(50, 2) + [-2, -2]
X_class1 = np.random.randn(50, 2) + [2, 2]
X = np.vstack([X_class0, X_class1])
y = np.hstack([np.zeros(50), np.ones(50)])

# Visualize different decision boundaries
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Multiple valid separators
ax = axes[0]
ax.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class -1', s=50, alpha=0.6)
ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class +1', s=50, alpha=0.6)

# Draw several separating lines
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
for w, b, alpha in [([1, 0], 0.5, 0.3), ([1, 1], 0, 0.3), ([0.5, 1], -1, 0.3)]:
    if w[1] != 0:
        x_plot = np.linspace(x_min, x_max, 100)
        y_plot = -(w[0] * x_plot + b) / w[1]
        ax.plot(x_plot, y_plot, 'g--', alpha=alpha, linewidth=1)

ax.set_title('Many Lines Can Separate the Data')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.set_xlim(x_min, x_max)
ax.set_ylim(-5, 5)

# Plot 2: Poor separator (close to points)
ax = axes[1]
ax.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class -1', s=50, alpha=0.6)
ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class +1', s=50, alpha=0.6)

# Draw poor separator
w_poor = np.array([0.5, 0.5])
b_poor = -0.2
x_plot = np.linspace(x_min, x_max, 100)
y_plot = -(w_poor[0] * x_plot + b_poor) / w_poor[1]
ax.plot(x_plot, y_plot, 'r-', linewidth=2, label='Poor separator')

ax.set_title('Poor Separator\n(Too Close to Points)')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.set_xlim(x_min, x_max)
ax.set_ylim(-5, 5)

# Plot 3: Maximum margin separator
ax = axes[2]
ax.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Class -1', s=50, alpha=0.6)
ax.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Class +1', s=50, alpha=0.6)

# Draw maximum margin separator
w_max = np.array([1, 1]) / np.sqrt(2)
b_max = 0
x_plot = np.linspace(x_min, x_max, 100)
y_plot = -(w_max[0] * x_plot + b_max) / w_max[1]
ax.plot(x_plot, y_plot, 'g-', linewidth=2, label='Max margin separator')

# Draw margin lines
margin = 1 / np.linalg.norm(w_max)
y_margin_pos = -(w_max[0] * x_plot + b_max + margin) / w_max[1]
y_margin_neg = -(w_max[0] * x_plot + b_max - margin) / w_max[1]
ax.plot(x_plot, y_margin_pos, 'g--', alpha=0.5, linewidth=1, label='Margin')
ax.plot(x_plot, y_margin_neg, 'g--', alpha=0.5, linewidth=1)

ax.set_title('Maximum Margin Separator\n(Best Robustness)')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.set_xlim(x_min, x_max)
ax.set_ylim(-5, 5)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("GEOMETRIC INTUITION")
print("="*70)
print("\nSupport Vector Machines maximize the margin (distance) between classes.")
print("\nKey insight: A decision boundary that is FAR from all points")
print("generalizes better than one that barely separates the data.")

<a name="mathematical-formulation"></a>
### Mathematical Formulation

For a binary classification problem, we want to find a decision boundary:

$$\mathbf{w}^T \mathbf{x} + b = 0$$

where $\mathbf{w}$ is the weight vector and $b$ is the bias.

**The margin** is the perpendicular distance from the decision boundary to the nearest data point. For a normalized weight vector, the margin is:

$$\text{margin} = \frac{1}{\|\mathbf{w}\|}$$

To maximize the margin, we minimize $\|\mathbf{w}\|^2$.

**For a linearly separable case**, we want:
- For class +1: $\mathbf{w}^T \mathbf{x}_i + b \geq 1$
- For class -1: $\mathbf{w}^T \mathbf{x}_i + b \leq -1$

Which can be written as:

$$y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 \quad \forall i$$

where $y_i \in \{-1, +1\}$.

The **hard-margin SVM optimization problem** is:

$$\begin{align}
\text{minimize} \quad &\frac{1}{2}\|\mathbf{w}\|^2 \\
\text{subject to} \quad &y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 \quad \forall i
\end{align}$$

In [ ]:
# Mathematical framework for SVM
print("\n" + "="*70)
print("MATHEMATICAL FORMULATION")
print("="*70)

print("\n1. DECISION BOUNDARY")
print("-" * 70)
print("   w^T x + b = 0")
print("\n   - w: weight vector (determines orientation)")
print("   - b: bias (determines position)")
print("   - Prediction: sign(w^T x + b)")

print("\n2. MARGIN")
print("-" * 70)
print("   margin = 1/||w||")
print("\n   - Larger margin = better generalization")
print("   - So we minimize ||w||^2 (equivalently, ||w||)")

print("\n3. CONSTRAINTS (for linearly separable data)")
print("-" * 70)
print("   For class +1: w^T x_i + b >= 1")
print("   For class -1: w^T x_i + b <= -1")
print("\n   Combined: y_i(w^T x_i + b) >= 1 for all i")
print("   where y_i ∈ {-1, +1}")

print("\n4. OPTIMIZATION PROBLEM (Hard-Margin SVM)")
print("-" * 70)
print("   Minimize:  (1/2) ||w||^2")
print("   Subject to: y_i(w^T x_i + b) >= 1 for all i")
print("\n   This is a convex quadratic programming problem.")

<a name="the-margin-and-support-vectors"></a>
### The Margin and Support Vectors

**Support vectors** are the data points that lie exactly on the margin boundary:
$$y_i(\mathbf{w}^T \mathbf{x}_i + b) = 1$$

These are the critical points that define the SVM. A key property:
- Only support vectors matter for the decision boundary
- Other points can be removed without changing the solution
- This sparsity is computationally efficient and prevents overfitting

<a name="lagrangian-duality"></a>
## Lagrangian Duality

<a name="primal-problem"></a>
### Primal Problem

The hard-margin SVM is a constrained optimization problem. We use the method of Lagrange multipliers.

**Lagrangian**:
$$L(\mathbf{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}\|\mathbf{w}\|^2 - \sum_{i=1}^{n} \alpha_i [y_i(\mathbf{w}^T \mathbf{x}_i + b) - 1]$$

where $\alpha_i \geq 0$ are Lagrange multipliers.

In [ ]:
# Demonstrate Lagrangian duality
print("\n" + "="*70)
print("LAGRANGIAN DUALITY")
print("="*70)

print("\n1. PRIMAL PROBLEM")
print("-" * 70)
print("   Minimize: (1/2)||w||^2")
print("   Subject to: y_i(w^T x_i + b) >= 1 for all i")

print("\n2. LAGRANGIAN FUNCTION")
print("-" * 70)
print("   L(w, b, α) = (1/2)||w||^2 - Σ α_i[y_i(w^T x_i + b) - 1]")
print("\n   where α_i >= 0 are Lagrange multipliers")

print("\n3. KKT CONDITIONS")
print("-" * 70)
print("   ∂L/∂w = 0:  w = Σ α_i y_i x_i")
print("   ∂L/∂b = 0:  Σ α_i y_i = 0")
print("   Complementary slackness: α_i[y_i(w^T x_i + b) - 1] = 0")
print("\n   Key insight: α_i > 0 only for support vectors")
print("   (points on the margin boundary)")

<a name="lagrangian-and-kkt-conditions"></a>
### Lagrangian and KKT Conditions

Taking derivatives and setting them to zero:

**Stationarity conditions**:
$$\frac{\partial L}{\partial \mathbf{w}} = \mathbf{w} - \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i = 0 \quad \Rightarrow \quad \mathbf{w} = \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i$$

$$\frac{\partial L}{\partial b} = -\sum_{i=1}^{n} \alpha_i y_i = 0 \quad \Rightarrow \quad \sum_{i=1}^{n} \alpha_i y_i = 0$$

**Complementary slackness**:
$$\alpha_i [y_i(\mathbf{w}^T \mathbf{x}_i + b) - 1] = 0$$

This means $\alpha_i > 0$ only when $y_i(\mathbf{w}^T \mathbf{x}_i + b) = 1$ (on the margin).

<a name="dual-problem"></a>
### Dual Problem

Substituting the stationarity conditions into the Lagrangian gives the **dual problem**:

$$\begin{align}
\text{maximize} \quad &\sum_{i=1}^{n} \alpha_i - \frac{1}{2}\sum_{i=1}^{n}\sum_{j=1}^{n} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T \mathbf{x}_j \\
\text{subject to} \quad &\sum_{i=1}^{n} \alpha_i y_i = 0 \\
& \alpha_i \geq 0 \quad \forall i
\end{align}$$

**Key observation**: The dual problem only involves dot products $\mathbf{x}_i^T \mathbf{x}_j$. This is where the kernel trick comes in!

<a name="the-kernel-trick"></a>
## The Kernel Trick

<a name="non-linear-separation"></a>
### Non-Linear Separation

Many real problems are not linearly separable. However, we can map the data to a higher-dimensional space where it becomes separable.

Instead of computing the explicit mapping $\phi(\mathbf{x})$, we can use a **kernel function** that computes the dot product in the high-dimensional space:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \phi(\mathbf{x}_i)^T \phi(\mathbf{x}_j)$$

This avoids the computational cost of explicit high-dimensional mappings.

In [ ]:
# Visualize non-linear separation with kernel trick
from matplotlib.patches import Circle

# Generate non-linearly separable data (XOR-like)
np.random.seed(42)
X_circle_in = np.random.randn(50, 2) * 0.5
X_circle_out = np.random.randn(100, 2) * 1.5
# Keep only outer ring
X_circle_out = X_circle_out[np.sum(X_circle_out**2, axis=1) > 1.5]
X_circle_out = X_circle_out[:50]

X_circle = np.vstack([X_circle_in, X_circle_out])
y_circle = np.hstack([np.ones(50), np.zeros(50)])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Original non-linearly separable data
ax = axes[0]
ax.scatter(X_circle[y_circle==0, 0], X_circle[y_circle==0, 1], 
          c='red', label='Class -1 (outer)', s=50, alpha=0.6)
ax.scatter(X_circle[y_circle==1, 0], X_circle[y_circle==1, 1], 
          c='blue', label='Class +1 (inner)', s=50, alpha=0.6)
ax.set_title('Original Space (Non-Linearly Separable)')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Conceptual transformation (2D representation of RBF kernel effect)
ax = axes[1]
# Use RBF kernel distance as a third dimension (conceptual)
center = np.array([0, 0])
r_original = np.sqrt(np.sum(X_circle**2, axis=1))

# RBF kernel effect: points become separable by distance
ax.scatter(r_original[y_circle==0], np.zeros(50), 
          c='red', label='Class -1 (outer)', s=100, alpha=0.6)
ax.scatter(r_original[y_circle==1], np.zeros(50), 
          c='blue', label='Class +1 (inner)', s=100, alpha=0.6)
ax.axvline(x=1.0, color='green', linestyle='--', linewidth=2, label='Linear separator')
ax.set_title('After RBF Kernel\n(Conceptual View)')
ax.set_xlabel('Transformed Feature')
ax.set_ylabel('')
ax.set_yticks([])
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("THE KERNEL TRICK")
print("="*70)
print("\nNon-linearly separable data in the original space becomes")
print("linearly separable after mapping to a higher-dimensional space.")
print("\nThe kernel trick computes dot products in high-dimensional space")
print("without explicitly computing the transformation.")

<a name="kernel-functions"></a>
### Kernel Functions

Common kernel functions:

**Linear kernel**:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \mathbf{x}_i^T \mathbf{x}_j$$

**Polynomial kernel** (degree $d$):
$$K(\mathbf{x}_i, \mathbf{x}_j) = (\mathbf{x}_i^T \mathbf{x}_j + c)^d$$

**Radial Basis Function (RBF) kernel**:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$

where $\gamma > 0$ is a parameter controlling the influence of single examples.

**Sigmoid kernel**:
$$K(\mathbf{x}_i, \mathbf{x}_j) = \tanh(\kappa \mathbf{x}_i^T \mathbf{x}_j + \theta)$$

In [ ]:
# Implement and visualize different kernels
def linear_kernel(X1, X2):
    """Linear kernel: K(x, y) = x^T y"""
    return np.dot(X1, X2.T)

def polynomial_kernel(X1, X2, degree=3, c=1):
    """Polynomial kernel: K(x, y) = (x^T y + c)^d"""
    return (np.dot(X1, X2.T) + c) ** degree

def rbf_kernel(X1, X2, gamma=1.0):
    """RBF kernel: K(x, y) = exp(-gamma * ||x - y||^2)"""
    # Efficient computation using broadcasting
    X1_sqnorm = np.sum(X1**2, axis=1, keepdims=True)
    X2_sqnorm = np.sum(X2**2, axis=1, keepdims=True).T
    X_dot = np.dot(X1, X2.T)
    return np.exp(-gamma * (X1_sqnorm + X2_sqnorm - 2 * X_dot))

# Create a simple 1D example to visualize kernel effects
x = np.linspace(-3, 3, 100).reshape(-1, 1)
x_samples = np.array([[-1], [1]])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Kernel Functions: How They Transform Distances', fontsize=14, fontweight='bold')

# Plot 1: Linear kernel
ax = axes[0, 0]
K_lin = linear_kernel(x, x_samples)
ax.plot(x, K_lin[:, 0], label='K(x, -1)', linewidth=2)
ax.plot(x, K_lin[:, 1], label='K(x, 1)', linewidth=2)
ax.scatter(x_samples, [0, 0], c='red', s=100, zorder=5, marker='x')
ax.set_title('Linear Kernel: K(x,y) = x·y')
ax.set_ylabel('Kernel Value')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Polynomial kernel (degree 3)
ax = axes[0, 1]
K_poly = polynomial_kernel(x, x_samples, degree=3, c=1)
ax.plot(x, K_poly[:, 0], label='K(x, -1)', linewidth=2)
ax.plot(x, K_poly[:, 1], label='K(x, 1)', linewidth=2)
ax.scatter(x_samples, [((-1)**3 + 1)**3, (1 + 1)**3], c='red', s=100, zorder=5, marker='x')
ax.set_title('Polynomial Kernel: K(x,y) = (x·y + 1)³')
ax.set_ylabel('Kernel Value')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: RBF kernel (gamma=1)
ax = axes[1, 0]
K_rbf = rbf_kernel(x, x_samples, gamma=1.0)
ax.plot(x, K_rbf[:, 0], label='K(x, -1)', linewidth=2)
ax.plot(x, K_rbf[:, 1], label='K(x, 1)', linewidth=2)
ax.scatter(x_samples, [1, 1], c='red', s=100, zorder=5, marker='x')
ax.set_title('RBF Kernel: K(x,y) = exp(-||x-y||²)')
ax.set_ylabel('Kernel Value')
ax.set_xlabel('x')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: RBF kernel (different gamma values)
ax = axes[1, 1]
for gamma in [0.1, 0.5, 1.0, 2.0]:
    K_rbf = rbf_kernel(x, x_samples[:1], gamma=gamma)
    ax.plot(x, K_rbf[:, 0], label=f'γ={gamma}', linewidth=2)
ax.scatter(x_samples[:1], [1], c='red', s=100, zorder=5, marker='x')
ax.set_title('RBF Kernel: Effect of γ Parameter')
ax.set_ylabel('Kernel Value')
ax.set_xlabel('x')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KERNEL FUNCTIONS MATHEMATICS")
print("="*70)
print("\nLinear kernel:     K(x,y) = x·y")
print("Polynomial kernel: K(x,y) = (x·y + c)^d")
print("RBF kernel:        K(x,y) = exp(-γ||x-y||²)")
print("Sigmoid kernel:    K(x,y) = tanh(κx·y + θ)")
print("\nEach kernel creates different geometric structure in feature space.")

<a name="mercers-theorem"></a>
### Mercer's Theorem

**Mercer's theorem** guarantees that if a function $K(\mathbf{x}_i, \mathbf{x}_j)$ is symmetric and positive semi-definite, then it corresponds to a valid dot product in some (possibly infinite-dimensional) Hilbert space.

This justifies using any symmetric positive semi-definite kernel function without explicitly knowing the transformation $\phi$.

<a name="soft-margin-svm"></a>
## Soft-Margin SVM

<a name="allowing-misclassification"></a>
### Allowing Misclassification

Real data is often not linearly separable (even in high dimensions). **Soft-margin SVM** allows some points to violate the margin.

We introduce **slack variables** $\xi_i \geq 0$:
$$y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 - \xi_i$$

The soft-margin optimization problem becomes:

$$\begin{align}
\text{minimize} \quad &\frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{n} \xi_i \\
\text{subject to} \quad &y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 - \xi_i \\
& \xi_i \geq 0
\end{align}$$

where $C > 0$ is a hyperparameter controlling the trade-off:
- **Large $C$**: Penalize misclassifications heavily (hard margin)
- **Small $C$**: Allow more misclassifications (soft margin)

<a name="hinge-loss"></a>
### Hinge Loss

The soft-margin problem can be equivalently written using the **hinge loss**:

$$\text{minimize} \quad \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \max(0, 1 - y_i(\mathbf{w}^T \mathbf{x}_i + b))$$

The **hinge loss** is:
$$\ell(y, \hat{y}) = \max(0, 1 - y\hat{y})$$

This is piecewise linear, making it suitable for gradient-based optimization.

In [ ]:
# Visualize soft-margin SVM and hinge loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Different margin violations
ax = axes[0]
# Generate slightly non-separable data
np.random.seed(42)
X_ns_class0 = np.random.randn(40, 2) + [-1.5, -1.5]
X_ns_class1 = np.random.randn(40, 2) + [1.5, 1.5]
# Add some outliers
X_ns_class0 = np.vstack([X_ns_class0, [[1.5, 1.5]]])
X_ns_class1 = np.vstack([X_ns_class1, [[-1.5, -1.5]]])

X_ns = np.vstack([X_ns_class0, X_ns_class1])
y_ns = np.hstack([np.zeros(41), np.ones(41)])

# Plot data
ax.scatter(X_ns[y_ns==0, 0], X_ns[y_ns==0, 1], c='red', label='Class -1', s=50, alpha=0.6)
ax.scatter(X_ns[y_ns==1, 0], X_ns[y_ns==1, 1], c='blue', label='Class +1', s=50, alpha=0.6)

# Draw approximate separator
w_approx = np.array([1, 1]) / np.sqrt(2)
b_approx = 0
x_plot = np.linspace(-4, 4, 100)
y_plot = -(w_approx[0] * x_plot + b_approx) / w_approx[1]
ax.plot(x_plot, y_plot, 'g-', linewidth=2, label='Decision boundary')

# Draw margins
margin = 1 / np.linalg.norm(w_approx)
y_margin_pos = -(w_approx[0] * x_plot + b_approx + margin) / w_approx[1]
y_margin_neg = -(w_approx[0] * x_plot + b_approx - margin) / w_approx[1]
ax.plot(x_plot, y_margin_pos, 'g--', alpha=0.5, linewidth=1, label='Margin')
ax.plot(x_plot, y_margin_neg, 'g--', alpha=0.5, linewidth=1)

ax.set_title('Soft-Margin SVM\n(Allows Margin Violations)')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)

# Plot 2: Hinge loss function
ax = axes[1]
z = np.linspace(-2, 3, 100)  # z = y * (w^T x + b)
hinge_loss = np.maximum(0, 1 - z)
zero_one_loss = (z <= 0).astype(float)

ax.plot(z, hinge_loss, 'b-', linewidth=2, label='Hinge loss: max(0, 1-z)')
ax.plot(z, zero_one_loss, 'r--', linewidth=2, label='0-1 loss')
ax.axvline(x=1, color='green', linestyle=':', alpha=0.5, label='Margin boundary')
ax.fill_between(z[z < 1], hinge_loss[z < 1], alpha=0.2, color='blue')

ax.set_title('Hinge Loss Function')
ax.set_xlabel('Margin: y(w^T x + b)')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 3)
ax.set_ylim(-0.1, 3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("SOFT-MARGIN SVM AND HINGE LOSS")
print("="*70)
print("\nSlack variables ξᵢ allow margin violations:")
print("  ξᵢ = 0: Point is on correct side of margin")
print("  0 < ξᵢ < 1: Point is within margin but correctly classified")
print("  ξᵢ > 1: Point is misclassified")
print("\nHinge loss: ℓ(z) = max(0, 1-z)")
print("  - Zero loss when margin is satisfied (z ≥ 1)")
print("  - Linear penalty for violations")
print("  - Continuous and piecewise-smooth (gradient exists except at z=1)")

<a name="from-scratch-implementation"></a>
## From-Scratch Implementation

<a name="linear-svm-with-gradient-descent"></a>
### Linear SVM with Gradient Descent

We implement soft-margin SVM using gradient descent on the hinge loss.

**Objective**:
$$L(\mathbf{w}, b) = \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{n} \max(0, 1 - y_i(\mathbf{w}^T \mathbf{x}_i + b))$$

**Gradient** (subgradient where hinge loss is not differentiable):

For each sample:
- If $y_i(\mathbf{w}^T \mathbf{x}_i + b) < 1$: 
  - $\frac{\partial L}{\partial \mathbf{w}} = \mathbf{w} - C y_i \mathbf{x}_i$
  - $\frac{\partial L}{\partial b} = -C y_i$

- Otherwise (margin satisfied):
  - $\frac{\partial L}{\partial \mathbf{w}} = \mathbf{w}$
  - $\frac{\partial L}{\partial b} = 0$

In [ ]:
class LinearSVM:
    """
    Support Vector Machine for binary classification using gradient descent.
    
    Uses soft-margin formulation with hinge loss:
    min (1/2)||w||^2 + C * sum(max(0, 1 - y_i * (w^T x_i + b)))
    """
    
    def __init__(self, learning_rate=0.01, C=1.0, epochs=100, batch_size=32):
        """
        Parameters:
        -----------
        learning_rate : float
            Learning rate for gradient descent
        C : float
            Regularization parameter (inverse of margin violation penalty)
        epochs : int
            Number of training epochs
        batch_size : int
            Batch size for mini-batch gradient descent
        """
        self.learning_rate = learning_rate
        self.C = C
        self.epochs = epochs
        self.batch_size = batch_size
        self.w = None
        self.b = 0
        self.losses = []
    
    def fit(self, X, y):
        """
        Train the SVM using gradient descent.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data
        y : array-like of shape (n_samples,)
            Target labels (-1 or +1)
        """
        n_samples, n_features = X.shape
        
        # Initialize weight vector
        self.w = np.zeros(n_features)
        self.b = 0
        
        # Training loop
        for epoch in range(self.epochs):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            epoch_loss = 0
            
            # Mini-batch gradient descent
            for i in range(0, n_samples, self.batch_size):
                batch_indices = indices[i:i+self.batch_size]
                X_batch = X[batch_indices]
                y_batch = y[batch_indices]
                
                # Compute predictions
                outputs = np.dot(X_batch, self.w) + self.b
                margins = y_batch * outputs
                
                # Compute loss
                hinge_losses = np.maximum(0, 1 - margins)
                batch_loss = 0.5 * np.dot(self.w, self.w) + self.C * np.mean(hinge_losses)
                epoch_loss += batch_loss
                
                # Compute gradients
                dw = self.w.copy()
                db = 0
                
                for j in range(len(batch_indices)):
                    if margins[j] < 1:
                        dw -= self.C * y_batch[j] * X_batch[j]
                        db -= self.C * y_batch[j]
                
                # Update parameters
                self.w -= self.learning_rate * dw / len(batch_indices)
                self.b -= self.learning_rate * db / len(batch_indices)
            
            self.losses.append(epoch_loss / (n_samples // self.batch_size))
            
            if (epoch + 1) % 20 == 0:
                print(f"Epoch {epoch+1}/{self.epochs}, Loss: {self.losses[-1]:.4f}")
        
        return self
    
    def predict(self, X):
        """
        Make predictions.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Data to predict on
        
        Returns:
        --------
        predictions : array of shape (n_samples,)
            Predicted class labels (-1 or +1)
        """
        return np.sign(np.dot(X, self.w) + self.b)
    
    def decision_function(self, X):
        """
        Compute the decision function (distance from hyperplane).
        """
        return np.dot(X, self.w) + self.b


print("\n" + "="*70)
print("LINEAR SVM IMPLEMENTATION")
print("="*70)
print("\nGradient descent optimization:")
print("  For each sample, check if margin is violated")
print("  If violated: gradient = w - C*y_i*x_i")
print("  If satisfied: gradient = w")
print("\nUpdate rule: w := w - learning_rate * gradient")

<a name="non-linear-svm-with-kernels"></a>
### Non-Linear SVM with Kernels

For non-linear SVM, we work in the dual space using kernel functions.

In [ ]:
class KernelSVM:
    """
    Kernel Support Vector Machine for non-linear classification.
    
    Uses the dual formulation with kernel trick.
    """
    
    def __init__(self, kernel='rbf', C=1.0, gamma=1.0, degree=3, c0=1, 
                 learning_rate=0.01, epochs=100):
        """
        Parameters:
        -----------
        kernel : str
            Type of kernel: 'linear', 'polynomial', 'rbf', 'sigmoid'
        C : float
            Regularization parameter
        gamma : float
            Kernel coefficient (for rbf and sigmoid)
        degree : int
            Degree for polynomial kernel
        c0 : float
            Constant term for polynomial kernel
        """
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.degree = degree
        self.c0 = c0
        self.learning_rate = learning_rate
        self.epochs = epochs
        
        self.X_train = None
        self.y_train = None
        self.alphas = None
        self.b = 0
        self.support_vectors = None
        self.support_vector_indices = None
    
    def _compute_kernel(self, X1, X2):
        """
        Compute kernel matrix K(X1, X2).
        
        K[i,j] = kernel(X1[i], X2[j])
        """
        if self.kernel == 'linear':
            return np.dot(X1, X2.T)
        
        elif self.kernel == 'polynomial':
            return (np.dot(X1, X2.T) + self.c0) ** self.degree
        
        elif self.kernel == 'rbf':
            # Efficient computation of exp(-gamma * ||x-y||^2)
            X1_sqnorm = np.sum(X1**2, axis=1, keepdims=True)
            X2_sqnorm = np.sum(X2**2, axis=1, keepdims=True).T
            X_dot = np.dot(X1, X2.T)
            distances_sq = X1_sqnorm + X2_sqnorm - 2 * X_dot
            return np.exp(-self.gamma * distances_sq)
        
        elif self.kernel == 'sigmoid':
            return np.tanh(self.gamma * np.dot(X1, X2.T) + self.c0)
        
        else:
            raise ValueError(f"Unknown kernel: {self.kernel}")
    
    def fit(self, X, y):
        """
        Train the Kernel SVM using gradient descent in dual space.
        """
        n_samples = X.shape[0]
        self.X_train = X
        self.y_train = y
        
        # Initialize Lagrange multipliers
        self.alphas = np.zeros(n_samples)
        
        # Compute kernel matrix
        K = self._compute_kernel(X, X)
        
        # Training
        for epoch in range(self.epochs):
            for i in range(n_samples):
                # Compute decision value for sample i
                decision = np.dot(self.alphas * y, K[i]) + self.b
                
                # Check if margin is violated
                if y[i] * decision < 1:
                    # Update alpha_i
                    alpha_old = self.alphas[i]
                    self.alphas[i] += self.learning_rate * y[i]
                    # Clip to [0, C]
                    self.alphas[i] = np.clip(self.alphas[i], 0, self.C)
                    
                    # Update bias
                    delta_alpha = self.alphas[i] - alpha_old
                    self.b += self.learning_rate * y[i] - delta_alpha * y[i] * K[i, i]
            
            if (epoch + 1) % 20 == 0:
                # Compute training accuracy
                predictions = self.predict(X)
                accuracy = np.mean(predictions == y)
                print(f"Epoch {epoch+1}/{self.epochs}, Accuracy: {accuracy:.4f}")
        
        # Identify support vectors (alphas > threshold)
        threshold = 1e-5
        self.support_vector_indices = np.where(self.alphas > threshold)[0]
        self.support_vectors = X[self.support_vector_indices]
        
        return self
    
    def predict(self, X):
        """
        Make predictions using the kernel function.
        """
        K = self._compute_kernel(X, self.X_train)
        decision = np.dot(K, self.alphas * self.y_train) + self.b
        return np.sign(decision)
    
    def decision_function(self, X):
        """
        Compute decision values.
        """
        K = self._compute_kernel(X, self.X_train)
        return np.dot(K, self.alphas * self.y_train) + self.b


print("\n" + "="*70)
print("KERNEL SVM IMPLEMENTATION")
print("="*70)
print("\nDual space optimization:")
print("  Work with Lagrange multipliers α_i")
print("  Compute kernel K(x_i, x_j) instead of explicit features")
print("  Decision function: f(x) = Σ α_i y_i K(x_i, x) + b")

<a name="visualization-and-interpretation"></a>
## Visualization and Interpretation

<a name="decision-boundaries"></a>
### Decision Boundaries

In [ ]:
# Train and visualize decision boundaries

# Create a simple 2D dataset for visualization
np.random.seed(42)
X_vis0 = np.random.randn(30, 2) + [-1.5, -1.5]
X_vis1 = np.random.randn(30, 2) + [1.5, 1.5]
X_vis = np.vstack([X_vis0, X_vis1])
y_vis = np.hstack([-np.ones(30), np.ones(30)])

# Train linear SVM
svm_lin = LinearSVM(learning_rate=0.01, C=1.0, epochs=100, batch_size=16)
svm_lin.fit(X_vis, y_vis)

# Create mesh for decision boundary visualization
x_min, x_max = X_vis[:, 0].min() - 1, X_vis[:, 0].max() + 1
y_min, y_max = X_vis[:, 1].min() - 1, X_vis[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                       np.linspace(y_min, y_max, 100))
Z = svm_lin.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.contourf(xx, yy, Z, levels=np.linspace(-3, 3, 20), cmap='RdBu', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)
ax.contour(xx, yy, Z, levels=[-1, 1], colors='black', linewidths=1, linestyles='--')

ax.scatter(X_vis[y_vis==-1, 0], X_vis[y_vis==-1, 1], c='red', label='Class -1', s=50, edgecolors='k')
ax.scatter(X_vis[y_vis==1, 0], X_vis[y_vis==1, 1], c='blue', label='Class +1', s=50, edgecolors='k')

ax.set_title('Linear SVM Decision Boundary\n(Black line: boundary, dashed lines: margin)')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Plot training loss
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(svm_lin.losses, linewidth=2, color='navy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('SVM Training Loss')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal weights: w = {svm_lin.w}")
print(f"Final bias: b = {svm_lin.b:.4f}")
print(f"Final loss: {svm_lin.losses[-1]:.4f}")

<a name="support-vectors"></a>
### Support Vectors

<a name="margin-visualization"></a>
### Margin Visualization

In [ ]:
# Visualize the learned margin
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot data
ax.scatter(X_vis[y_vis==-1, 0], X_vis[y_vis==-1, 1], c='red', label='Class -1', s=50, alpha=0.6)
ax.scatter(X_vis[y_vis==1, 0], X_vis[y_vis==1, 1], c='blue', label='Class +1', s=50, alpha=0.6)

# Plot decision boundary and margin
x_plot = np.linspace(x_min, x_max, 100)
# Decision boundary: w^T x + b = 0
if svm_lin.w[1] != 0:
    y_boundary = -(svm_lin.w[0] * x_plot + svm_lin.b) / svm_lin.w[1]
    ax.plot(x_plot, y_boundary, 'g-', linewidth=2, label='Decision boundary')
    
    # Margin boundaries: w^T x + b = ±1
    margin = 1 / np.linalg.norm(svm_lin.w)
    y_pos_margin = -(svm_lin.w[0] * x_plot + svm_lin.b + 1) / svm_lin.w[1]
    y_neg_margin = -(svm_lin.w[0] * x_plot + svm_lin.b - 1) / svm_lin.w[1]
    
    ax.plot(x_plot, y_pos_margin, 'g--', linewidth=1, alpha=0.7, label='Margin boundary')
    ax.plot(x_plot, y_neg_margin, 'g--', linewidth=1, alpha=0.7)
    
    # Shade margin region
    ax.fill_between(x_plot, y_neg_margin, y_pos_margin, alpha=0.1, color='green')

ax.set_title(f'Margin Width = 2/||w|| = {2/np.linalg.norm(svm_lin.w):.4f}')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("MARGIN ANALYSIS")
print("="*70)
print(f"\nWeight vector norm: ||w|| = {np.linalg.norm(svm_lin.w):.4f}")
print(f"Margin width: 2/||w|| = {2/np.linalg.norm(svm_lin.w):.4f}")
print(f"\nLarger ||w|| → smaller margin → risk of underfitting")
print(f"Smaller ||w|| → larger margin → better generalization")

<a name="practical-application"></a>
## Practical Application

<a name="binary-classification-on-breast-cancer-data"></a>
### Binary Classification on Breast Cancer Data

In [ ]:
# Load and prepare data
data = load_breast_cancer()
X_bc = data.data
y_bc = data.target

# Convert labels to -1 and +1
y_bc = np.where(y_bc == 1, 1, -1)

# Standardize features
scaler = StandardScaler()
X_bc = scaler.fit_transform(X_bc)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

print("\n" + "="*70)
print("BREAST CANCER DATASET")
print("="*70)
print(f"\nTotal samples: {X_bc.shape[0]}")
print(f"Features per sample: {X_bc.shape[1]}")
print(f"Class distribution: {np.sum(y_bc == 1)} positive, {np.sum(y_bc == -1)} negative")
print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Train linear SVM on breast cancer data
print("\nTraining Linear SVM...")
svm_cancer = LinearSVM(learning_rate=0.001, C=1.0, epochs=100, batch_size=16)
svm_cancer.fit(X_train, y_train)

# Make predictions
y_pred_train = svm_cancer.predict(X_train)
y_pred_test = svm_cancer.predict(X_test)

# Evaluate
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"\n" + "="*70)
print("LINEAR SVM RESULTS")
print("="*70)
print(f"\nTraining accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"\nConfusion Matrix (Test Set):")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)
print(f"\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test, target_names=['Negative', 'Positive']))

<a name="performance-analysis"></a>
### Performance Analysis

In [ ]:
# Compare different regularization parameters
C_values = [0.1, 0.5, 1.0, 5.0, 10.0]
train_accs = []
test_accs = []

for C in C_values:
    svm = LinearSVM(learning_rate=0.001, C=C, epochs=100, batch_size=16)
    svm.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, svm.predict(X_train))
    test_acc = accuracy_score(y_test, svm.predict(X_test))
    
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    print(f"C={C:5.1f}: Train={train_acc:.4f}, Test={test_acc:.4f}")

# Plot results
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(C_values, train_accs, 'o-', linewidth=2, markersize=8, label='Training accuracy')
ax.plot(C_values, test_accs, 's-', linewidth=2, markersize=8, label='Test accuracy')
ax.set_xlabel('Regularization parameter C')
ax.set_ylabel('Accuracy')
ax.set_title('SVM Performance vs Regularization')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.85, 1.0])

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("REGULARIZATION EFFECT")
print("="*70)
print("\nLarge C: Penalize margin violations heavily")
print("  → Lower training error, higher test error (overfitting)")
print("\nSmall C: Allow more margin violations")
print("  → Larger margin, better generalization")

<a name="conclusion"></a>
## Conclusion

<a name="key-insights"></a>
### Key Insights

1. **Maximum margin principle** provides a principled approach to finding robust decision boundaries

2. **Lagrangian duality** transforms a constrained optimization problem into one that only requires dot products of data points

3. **The kernel trick** enables non-linear classification without explicitly computing high-dimensional transformations

4. **Soft-margin formulation** with hinge loss makes SVMs practical for real (non-separable) data

5. **Support vectors** are the key points that determine the decision boundary — other points can be discarded

6. **Regularization parameter C** controls the bias-variance trade-off

<a name="when-to-use-svms"></a>
### When to Use SVMs

**SVMs are particularly good for:**
- Small to medium-sized datasets (< 1 million samples)
- Problems where interpretability isn't critical
- High-dimensional data (text classification, image recognition)
- Binary classification (though multi-class extensions exist)
- Problems where you have strong priors about non-linearity

**SVMs are less suitable for:**
- Very large datasets (training time is O(n²) or worse)
- Real-time prediction requirements (kernel computations are expensive)
- Highly imbalanced datasets (unless using weighted SVMs)
- When probabilistic outputs are needed (SVMs give hard predictions)

<a name="further-reading"></a>
### Further Reading

**Foundational Papers:**
- Vapnik, V. N. (1995). "The Nature of Statistical Learning Theory"
- Boser, B. E., Guyon, I. M., & Vapnik, V. N. (1992). "A training algorithm for optimal margin classifiers"
- Schölkopf, B., & Smola, A. J. (2002). "Learning with Kernels"

**Comprehensive References:**
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). "The Elements of Statistical Learning" (Chapter 12)
- Christianini, N., & Shawe-Taylor, J. (2000). "An Introduction to Support Vector Machines and Other Kernel-Based Learning Methods"
- Stanford CS229 notes on SVMs: http://cs229.stanford.edu/